In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

# Read Customer Dimension
dim_customer = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_customer.csv")
)

# Read Account Dimension
dim_account = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_account.csv")
)

# Read Branch Dimension
dim_branch = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_branch.csv")
)

# Read Channel Dimension
dim_channel = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_channel.csv")
)

# Read Date Dimension
dim_date = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_date.csv")
)

# Read Employee Dimension
dim_employee = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_employee.csv")
)

# Read Product Dimension
dim_product = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_product.csv")
)

StatementMeta(, a3d146e8-12e3-4785-91a2-b3850b43d3a2, 5, Finished, Available, Finished, False)

In [4]:
# Write dimension tables to Lakehouse as Delta tables

dim_customer.write.mode("overwrite").format("delta").saveAsTable("dim_customer")
dim_account.write.mode("overwrite").format("delta").saveAsTable("dim_account")
dim_branch.write.mode("overwrite").format("delta").saveAsTable("dim_branch")
dim_channel.write.mode("overwrite").format("delta").saveAsTable("dim_channel")
dim_date.write.mode("overwrite").format("delta").saveAsTable("dim_date")
dim_employee.write.mode("overwrite").format("delta").saveAsTable("dim_employee")
dim_product.write.mode("overwrite").format("delta").saveAsTable("dim_product")

StatementMeta(, a3d146e8-12e3-4785-91a2-b3850b43d3a2, 6, Finished, Available, Finished, False)

In [6]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# Incoming latest customer data from shortcut
incoming_customer_df = (
    spark.read
    .option("header", "true")
    .csv("Files/dimensionfiles/dim_customer.csv")
)

# Set correct data type for key
incoming_customer_df = incoming_customer_df.withColumn(
    "customer_key", col("customer_key").cast("int")
)

# Existing target Delta table
target_table = DeltaTable.forName(spark, "dim_customer")

# SCD Type 1 Merge
target_table.alias("target").merge(
    incoming_customer_df.alias("source"),
    "target.customer_key = source.customer_key"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

StatementMeta(, a3d146e8-12e3-4785-91a2-b3850b43d3a2, 8, Finished, Available, Finished, False)

In [7]:
df = spark.sql("SELECT * FROM BankLakehouse.dbo.fact_transactions LIMIT 1000")
display(df)

StatementMeta(, a3d146e8-12e3-4785-91a2-b3850b43d3a2, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 226cd3f1-d58b-4ddc-b03a-6509f0907af3)

In [5]:
df = spark.sql("SELECT * FROM BankLakehouse.dbo.dim_customer LIMIT 1000")
display(df)

StatementMeta(, a3d146e8-12e3-4785-91a2-b3850b43d3a2, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4688f156-e438-4618-9343-fca38502470a)